In [21]:
import pandas as pd

# LOAD DATA
job_listings = pd.read_csv("../data/raw/skill demand dataset/job_postings.csv")
job_skills = pd.read_csv("../data/raw/skill demand dataset/job_skills.csv")
students = pd.read_excel("../data/raw/student_career_dataset/Career Dataset.xlsx")

In [22]:

# fix column name for merge
job_listings.rename(columns={"Link": "job_link"}, inplace=True)

# MERGE JOB + SKILLS
merged_jobs = pd.merge(job_listings, job_skills, on="job_link")
print("Columns after merge:")
print(merged_jobs.columns)

Columns after merge:
Index(['job_link', 'last_processed_time', 'last_status', 'got_summary',
       'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location',
       'first_seen', 'search_city', 'search_country', 'search_position',
       'job_level', 'job_type', 'job_skills'],
      dtype='str')


In [23]:
# CLEAN INDUSTRY SKILLS (job_skills column)
# -------------------------
merged_jobs['job_skills'] = merged_jobs['job_skills'].astype(str)

merged_jobs['job_skills'] = merged_jobs['job_skills'].str.lower()
merged_jobs['job_skills'] = merged_jobs['job_skills'].str.strip()

merged_jobs['job_skills'] = merged_jobs['job_skills'].str.replace('&', 'and')
merged_jobs['job_skills'] = merged_jobs['job_skills'].str.replace('[^a-zA-Z0-9, ]', '', regex=True)

# split multiple skills
merged_jobs['job_skills'] = merged_jobs['job_skills'].str.split(',')
merged_jobs = merged_jobs.explode('job_skills')

merged_jobs['job_skills'] = merged_jobs['job_skills'].str.strip()

In [24]:
# CLEAN STUDENT DATA
# -------------------------
students['Skill'] = students['Skill'].astype(str)

students['Skill'] = students['Skill'].str.split(',')
students = students.explode('Skill')

students['Skill'] = students['Skill'].str.lower()
students['Skill'] = students['Skill'].str.strip()

students['Skill'] = students['Skill'].str.replace('&', 'and')
students['Skill'] = students['Skill'].str.replace('[^a-zA-Z0-9 ]', '', regex=True)


In [25]:
# STANDARD SKILL NAMES
# -------------------------
skill_map = {
    "ai": "artificial intelligence",
    "ml": "machine learning",
    "ai ml": "machine learning",
    "ai and ml": "machine learning",
    "artificial intelligence ai": "artificial intelligence",
    "python programming": "python",
    "data analytics": "data analysis",
    "data science": "data science",
    "deep learning": "machine learning",
    "nlp": "natural language processing"
}

merged_jobs['job_skills'] = merged_jobs['job_skills'].replace(skill_map)
students['Skill'] = students['Skill'].replace(skill_map)

In [26]:
# SAVE CLEAN FILES
# -------------------------
merged_jobs.to_csv("../data/processed/industry_cleaned.csv", index=False)
students.to_csv("../data/processed/student_cleaned.csv", index=False)

print("Data Cleaning Completed Successfully!")

Data Cleaning Completed Successfully!
